# MScFE 600 Financial Data — Group Work Project 1

## Technical indicators and stock-market direction

**Selected fund:** iShares Core S&P 500 ETF (IVV)  
**Study period:** December 2009 to January 2020

The analysis follows the questions in the assignment and the approach described by Sagaceta-Mejía, Sánchez-Gutiérrez, and Fresán-Figueroa (2024). It covers the data and fund, summarizes the feature-selection methods in the paper, and applies Pearson correlation to a smaller set of technical indicators.

The notebook also includes a Search Volume Index (SVI) example for groups completing Part 2. The market-data analysis and the SVI example are kept separate because they address different questions.

## Before running the notebook

Run the cells in order. The price history is downloaded from Yahoo Finance, so an internet connection is needed. The requested end date is exclusive; using `2020-01-02` includes the final trading session of 2019.

The paper uses 216 technical indicators and reports ten-fold cross-validation. This notebook uses eight indicators and five expanding-window time-series folds with Pearson correlation as the feature-selection method. It is a smaller, transparent replication exercise rather than an exact reproduction of every experiment in the paper.

The feature ranking and scaling are fitted within each training fold. Only earlier indicator values are used to predict the next day's class. This preserves the time order and reduces look-ahead leakage.

In [ ]:
%pip install -q yfinance scikit-learn seaborn

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TICKER = "IVV"
START_DATE = "2009-12-12"
END_DATE = "2020-01-02"  # Yahoo Finance excludes this date
N_SPLITS = 5
TOP_N_FEATURES = 5

PRICE_COLUMNS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

In [ ]:
def download_price_data(ticker, start, end):
    """Download and clean the daily OHLCV history for one ETF."""
    prices = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        actions=False,
        progress=False,
        threads=False,
    )

    if prices.empty:
        raise RuntimeError(
            "No prices were downloaded. Check the internet connection, ticker, and date range."
        )

    # yfinance may return a MultiIndex even when only one ticker is requested.
    if isinstance(prices.columns, pd.MultiIndex):
        field_names = set(PRICE_COLUMNS)
        prices.columns = [
            next((str(part) for part in column if str(part) in field_names), str(column[0]))
            for column in prices.columns
        ]

    prices = prices.loc[:, ~prices.columns.duplicated()].copy()
    missing = [column for column in PRICE_COLUMNS if column not in prices.columns]
    if missing:
        raise ValueError(f"Yahoo Finance data is missing these fields: {missing}")

    prices = prices[PRICE_COLUMNS].sort_index()
    prices.index = pd.to_datetime(prices.index)
    prices.index.name = "Date"
    return prices.dropna(subset=PRICE_COLUMNS)


data = download_price_data(TICKER, START_DATE, END_DATE)

print(f"Fund: {TICKER}")
print(f"Observations: {len(data):,}")
print(f"First date: {data.index.min().date()}")
print(f"Last date:  {data.index.max().date()}")
display(data.head())

## Part 1 — Q1 and Q2: Data and security

### Q1. Data used in the paper

The authors use daily market data for three ETFs: iShares MSCI Chile ETF (ECH), iShares MSCI Brazil ETF (EWZ), and iShares Core S&P 500 ETF (IVV). The raw variables obtained from Yahoo Finance are Open, High, Low, Close, Adjusted Close, and Volume. These are numerical time-series observations. The authors derive 210 technical indicators from the raw fields, giving 216 features in total.

Indicators describe aspects of price and trading activity that may not be obvious from a single closing price. For example, moving averages summarize recent direction, RSI measures momentum, and Bollinger Bands describe price movement relative to recent volatility. These features can help a classifier identify market patterns. They may also be highly correlated with one another, and past patterns do not guarantee future price movements.

### Q2. Fund description — IVV

IVV is an exchange-traded fund designed to track the S&P 500 Index, which represents large U.S. companies. It provides exposure to a broad basket of companies rather than to a single stock. Its price can respond to changes in company earnings, interest rates, economic expectations, and the prices of the companies in the index.

The table and chart below summarize IVV over the paper's sample period. They report the historical sample, not the fund's current price or assets under management. Use the output to prepare the required one-page fund description in the report; the assignment sets a strict one-page limit.

**Why classification rather than regression?** The authors predict the direction of the opening price, rather than its exact future value. A directional class can be directly related to a rise or fall in price. Regression would instead estimate a continuous quantity such as the next-day return.

Two alternative ways to define the class are:
1. **Adjusted-close direction:** label a day as +1 if the adjusted close is higher than the previous day's adjusted close, and −1 otherwise.
2. **Three classes based on return:** label a sufficiently positive return as +1, a sufficiently negative return as −1, and a return within a pre-defined band around zero as 0.

The thresholds for a three-class label should be decided before evaluation. The choice of label changes the prediction problem, so results from different definitions are not directly interchangeable.

In [ ]:
def summarize_fund(prices):
    """Return sample-period price and risk statistics for the selected fund."""
    adjusted_close = prices["Adj Close"].dropna()
    daily_returns = adjusted_close.pct_change().dropna()
    years = (adjusted_close.index[-1] - adjusted_close.index[0]).days / 365.25
    wealth_index = adjusted_close / adjusted_close.iloc[0]
    drawdown = wealth_index / wealth_index.cummax() - 1

    return pd.DataFrame({
        "Measure": [
            "Trading observations",
            "First date",
            "Last date",
            "First adjusted close (USD)",
            "Last adjusted close (USD)",
            "Total adjusted-price return",
            "Annualized return (CAGR)",
            "Annualized volatility",
            "Maximum drawdown",
            "Mean daily return",
            "Median daily return",
        ],
        "Value": [
            len(prices),
            prices.index.min().date().isoformat(),
            prices.index.max().date().isoformat(),
            f"${adjusted_close.iloc[0]:.2f}",
            f"${adjusted_close.iloc[-1]:.2f}",
            f"{adjusted_close.iloc[-1] / adjusted_close.iloc[0] - 1:.2%}",
            f"{(adjusted_close.iloc[-1] / adjusted_close.iloc[0]) ** (1 / years) - 1:.2%}",
            f"{daily_returns.std(ddof=1) * np.sqrt(252):.2%}",
            f"{drawdown.min():.2%}",
            f"{daily_returns.mean():.4%}",
            f"{daily_returns.median():.4%}",
        ],
    })


display(summarize_fund(data))

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(data.index, data["Adj Close"], linewidth=1.5)
ax.set_title("IVV adjusted close, December 2009–December 2019")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted close (USD)")
fig.tight_layout()
plt.show()

display(data["Open"].describe().rename("Open price (USD)").to_frame())

## Q3. Methodology: reorganizing Sections 2 and 3

A clearer structure separates the data description from the methods used to select features and train the model.

### Proposed Section 2 — Data
- **2.1 ETFs and sample period:** ECH, EWZ, and IVV; 12 December 2009 to 1 January 2020.
- **2.2 Data source and raw fields:** daily Open, High, Low, Close, Adjusted Close, and Volume from Yahoo Finance.
- **2.3 Technical indicators:** construction of 210 derived indicators from the six raw fields.
- **2.4 Class definition:** the opening-price direction, coded +1 when today's opening price is higher than yesterday's and −1 otherwise.
- **2.5 Data preparation:** normalization and the treatment of missing observations.
- **2.6 Initial exploration:** summary statistics and the distribution of the response classes.

### Proposed Section 3 — Methodology
- **3.1 CRISP-DM workflow:** business understanding, data understanding, preparation, modeling, evaluation, and presentation.
- **3.2 Statistical feature-selection methods:** Pearson correlation and statistical rankings based on measures such as variance, skewness, kurtosis, mean absolute difference, and dispersion ratio.
- **3.3 Model-based methods:** LASSO, tree-based feature selection, and Principal Feature Analysis.
- **3.4 Feature-set construction:** form subsets from indicators selected across the ranking procedures and compare overlaps with Jaccard distance.
- **3.5 Neural-network classifier:** train the multilayer perceptron (MLP) using the selected indicators.
- **3.6 Evaluation:** cross-validation, classification accuracy, and training time.

Descriptive statistics such as means and standard deviations belong in the data exploration section. Pearson correlation measures the association between variables; when it is used to rank candidate predictors, it also serves as a feature-selection method. LASSO is model-based because it estimates coefficients with a penalty that can shrink some of them to zero.

The authors compare several selection methods, then evaluate feature subsets using an MLP. Their results show that a small subset can reduce training time while retaining or improving accuracy. The notebook below uses Pearson correlation because the assignment allows a simpler metric for the replication.

## Q4. Features, methods, and models

A **feature** is an input variable supplied to a model. Examples include RSI, MACD, rolling volatility, and a rate-of-change indicator.

A **method** is a procedure used to prepare or analyze data. Examples are correlation-based ranking, LASSO, normalization, and cross-validation.

A **model** is the fitted relationship between the inputs and the outcome. In the paper, the prediction model is a multilayer perceptron neural network.

Features can be grouped into raw market variables (prices and volume), engineered technical indicators, lagged indicators, and external or alternative data such as search interest or news measures. The response class is the outcome being predicted; it should not be included among the input features.

Selecting indicators can remove redundant information and reduce computation. It is important to select them using training data only. Choosing features after looking at the entire dataset would allow information from the test period to affect model development and would make the reported test performance unreliable.

## Q5. Cross-validation and optimization

**Cross-validation** tests how a model performs across several data partitions instead of relying on a single split. **K-fold cross-validation** divides observations into *k* folds and evaluates each fold once. In ordinary shuffled K-fold, training observations may occur later in time than validation observations. That is a problem for forecasting. This notebook therefore uses expanding-window time-series folds, in which each test window follows its training window.

**Jaccard distance** compares two sets. For feature sets A and B,

`d_J(A,B) = 1 - |A ∩ B| / |A ∪ B|`

A value of 0 means the sets are identical; a value of 1 means they share no features. Here it measures whether the five highest-ranked features remain similar from one fold to the next.

Two other distance measures are:
- **Euclidean distance:** the square root of the sum of squared differences between numeric vectors.
- **Manhattan distance:** the sum of absolute differences between numeric vectors.

Jaccard distance is appropriate for comparing feature membership. Euclidean and Manhattan distances compare numeric values and can change substantially with scale.

In the paper, a useful solution is a compact feature set that produces good cross-validated accuracy with less computation. For IVV, Table 6 reports 71.10% accuracy with all 216 features and 78.46% with the selected nine features. Those are published results; the results from the smaller experiment below will be calculated separately.

### Financial problem and practical findings

The financial problem is to classify ETF price direction using technical indicators while reducing the number of inputs the model must process. ECH and EWZ represent emerging markets, while IVV provides a U.S. large-cap comparison. Differences in liquidity, volatility, trading conditions, and market composition can affect how well indicators work, so results for one fund should not automatically be transferred to another.

The paper reports that feature selection improved IVV's accuracy from 71.10% to 78.46% in its early-stopping experiment and reduced training time from 53.2 to 8.68 seconds. The selected IVV indicators include Bollinger Band Percent, Balance of Power, increasing/decreasing indicators, a J-based indicator, PVR, TTM trend, Williams %R, and Stochastic RSI. The authors' numbers describe their own experiment and sample; they are not performance claims for a live trading strategy.

In [ ]:
FEATURE_COLUMNS = [
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Hist",
    "Close_MA20_Ratio",
    "BB_Width_20",
    "Rolling_Volatility_20",
    "ROC_10",
]


def add_technical_indicators(prices):
    """Calculate eight technical indicators and the paper's direction label."""
    frame = prices.copy()
    close = frame["Close"]

    change = close.diff()
    average_gain = change.clip(lower=0).ewm(
        alpha=1 / 14, min_periods=14, adjust=False
    ).mean()
    average_loss = (-change.clip(upper=0)).ewm(
        alpha=1 / 14, min_periods=14, adjust=False
    ).mean()

    relative_strength = average_gain / average_loss.replace(0, np.nan)
    frame["RSI_14"] = 100 - (100 / (1 + relative_strength))
    frame.loc[(average_loss == 0) & (average_gain > 0), "RSI_14"] = 100

    ema_12 = close.ewm(span=12, min_periods=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, min_periods=26, adjust=False).mean()
    frame["MACD"] = ema_12 - ema_26
    frame["MACD_Signal"] = frame["MACD"].ewm(
        span=9, min_periods=9, adjust=False
    ).mean()
    frame["MACD_Hist"] = frame["MACD"] - frame["MACD_Signal"]

    moving_average = close.rolling(20, min_periods=20).mean()
    rolling_std = close.rolling(20, min_periods=20).std(ddof=0)
    upper_band = moving_average + 2 * rolling_std
    lower_band = moving_average - 2 * rolling_std
    frame["Close_MA20_Ratio"] = close / moving_average - 1
    frame["BB_Width_20"] = (upper_band - lower_band) / moving_average

    log_return = np.log(frame["Adj Close"] / frame["Adj Close"].shift(1))
    frame["Rolling_Volatility_20"] = (
        log_return.rolling(20, min_periods=20).std() * np.sqrt(252)
    )
    frame["ROC_10"] = close.pct_change(10)

    # Gamma is +1 when today's open is above yesterday's, and -1 otherwise.
    frame["Gamma"] = np.where(
        frame["Open"] > frame["Open"].shift(1), 1, -1
    )
    return frame


features_data = add_technical_indicators(data)

# Use information available at t-1 to predict the class at t.
model_data = features_data[FEATURE_COLUMNS].shift(1)
model_data["Gamma"] = features_data["Gamma"]
model_data = model_data.replace([np.inf, -np.inf], np.nan).dropna()

X = model_data[FEATURE_COLUMNS].astype(float)
y = model_data["Gamma"].astype(int)

if len(model_data) < 100:
    raise ValueError("Too few usable observations for the planned validation.")

print(f"Usable observations: {len(model_data):,}")
print(f"Number of input features: {len(FEATURE_COLUMNS)}")
display(y.value_counts().rename(index={-1: "Gamma = -1", 1: "Gamma = +1"}).to_frame("Count"))
display((y.value_counts(normalize=True) * 100).rename(index={-1: "Gamma = -1", 1: "Gamma = +1"}).to_frame("Share (%)").round(2))
display(model_data.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(data.index, data["Adj Close"], linewidth=1.4)
axes[0].set_title("IVV adjusted close")
axes[0].set_ylabel("USD")

gamma = features_data.loc[model_data.index, "Gamma"]
axes[1].plot(gamma.index, gamma.cumsum(), linewidth=1.3)
axes[1].set_title("Cumulative sum of the direction label")
axes[1].set_ylabel("Cumulative Gamma")
axes[1].set_xlabel("Date")

fig.tight_layout()
plt.show()

## Step 3 — Replication using Pearson correlation

The replication uses five expanding-window folds. In each fold, absolute Pearson correlation with the training labels ranks the eight candidate indicators. The MLP is then evaluated twice: once using all eight indicators and once using the top five. A majority-class classifier provides a simple baseline.

Feature selection and scaling are fitted using the training period only. Test periods are kept in date order. The design is intentionally smaller than the paper's 216-feature, ten-fold experiment, so the results should be described as a partial replication.

In [ ]:
def build_classifier():
    """Create a standardized MLP pipeline."""
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=0.001,
            max_iter=500,
            random_state=RANDOM_STATE,
        ),
    )


def evaluate_walk_forward(X, y, n_splits=5, top_n=5):
    """Evaluate baseline, full feature set, and correlation-selected features."""
    splitter = TimeSeriesSplit(n_splits=n_splits)
    fold_rows = []
    ranking_rows = []
    prediction_frames = []
    previous_features = None

    for fold, (train_index, test_index) in enumerate(splitter.split(X), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        correlations = X_train.join(y_train.rename("Gamma")).corr()["Gamma"].drop("Gamma")
        ranking = correlations.abs().sort_values(ascending=False)
        selected_features = ranking.head(top_n).index.tolist()

        for feature, score in ranking.items():
            ranking_rows.append({
                "Fold": fold,
                "Feature": feature,
                "Absolute Pearson correlation": float(score),
            })

        baseline = DummyClassifier(strategy="most_frequent")
        baseline.fit(X_train, y_train)
        baseline_prediction = baseline.predict(X_test)

        all_features_model = build_classifier()
        selected_features_model = build_classifier()
        all_features_model.fit(X_train, y_train)
        selected_features_model.fit(X_train[selected_features], y_train)

        all_features_prediction = all_features_model.predict(X_test)
        selected_prediction = selected_features_model.predict(X_test)

        current_features = set(selected_features)
        if previous_features is None:
            jaccard_distance = np.nan
        else:
            union = current_features | previous_features
            intersection = current_features & previous_features
            jaccard_distance = 1 - len(intersection) / len(union) if union else 0.0

        fold_rows.append({
            "Fold": fold,
            "Test start": X_test.index.min().date().isoformat(),
            "Test end": X_test.index.max().date().isoformat(),
            "Baseline accuracy": accuracy_score(y_test, baseline_prediction),
            "All 8 features accuracy": accuracy_score(y_test, all_features_prediction),
            "Top 5 accuracy": accuracy_score(y_test, selected_prediction),
            "Top 5 balanced accuracy": balanced_accuracy_score(y_test, selected_prediction),
            "Top 5 precision": precision_score(y_test, selected_prediction, pos_label=1, zero_division=0),
            "Top 5 recall": recall_score(y_test, selected_prediction, pos_label=1, zero_division=0),
            "Top 5 F1": f1_score(y_test, selected_prediction, pos_label=1, zero_division=0),
            "Feature-set Jaccard distance": jaccard_distance,
            "Selected features": ", ".join(selected_features),
        })

        prediction_frames.append(pd.DataFrame({
            "Actual Gamma": y_test,
            "Predicted Gamma": pd.Series(selected_prediction, index=y_test.index),
            "Fold": fold,
        }, index=y_test.index))

        previous_features = current_features
        print(
            f"Fold {fold}: {X_test.index.min().date()} to {X_test.index.max().date()} "
            f"({len(X_test)} test observations)"
        )

    return (
        pd.DataFrame(fold_rows),
        pd.concat(prediction_frames).sort_index(),
        pd.DataFrame(ranking_rows),
    )


results_df, oos_predictions, feature_rankings = evaluate_walk_forward(
    X, y, n_splits=N_SPLITS, top_n=TOP_N_FEATURES
)

display(results_df.round(4))

metric_columns = [
    "Baseline accuracy",
    "All 8 features accuracy",
    "Top 5 accuracy",
    "Top 5 balanced accuracy",
    "Top 5 precision",
    "Top 5 recall",
    "Top 5 F1",
]
display(results_df[metric_columns].mean().rename("Mean across folds").to_frame().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

performance = results_df.melt(
    id_vars="Fold",
    value_vars=["Baseline accuracy", "All 8 features accuracy", "Top 5 accuracy"],
    var_name="Approach",
    value_name="Accuracy",
)
sns.lineplot(data=performance, x="Fold", y="Accuracy", hue="Approach", marker="o", ax=axes[0])
axes[0].set_title("Out-of-sample accuracy by fold")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)

mean_correlations = (
    feature_rankings.groupby("Feature")["Absolute Pearson correlation"]
    .mean()
    .sort_values()
)
mean_correlations.plot(kind="barh", ax=axes[1])
axes[1].set_title("Mean absolute feature correlation")
axes[1].set_xlabel("Absolute Pearson correlation")
axes[1].set_ylabel("Feature")

fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    X.corr(),
    annot=True,
    fmt=".2f",
    center=0,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Correlation among input features")
fig.tight_layout()
plt.show()

confusion = confusion_matrix(
    oos_predictions["Actual Gamma"],
    oos_predictions["Predicted Gamma"],
    labels=[-1, 1],
)
display(pd.DataFrame(
    confusion,
    index=["Actual -1", "Actual +1"],
    columns=["Predicted -1", "Predicted +1"],
))

ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=["Gamma = -1", "Gamma = +1"],
).plot()
plt.title("Out-of-sample confusion matrix")
plt.tight_layout()
plt.show()

## Results: what to report

Use the tables and figures above to describe what happened in this run. The main comparisons are the majority-class baseline, the model with all eight indicators, and the model with the five indicators selected in each training fold.

Report the mean scores alongside the individual folds. Balanced accuracy and the class-specific metrics help reveal whether a model favors one direction. The Jaccard distance describes whether the selected features remain stable between adjacent folds; it is not a measure of classification accuracy.

This notebook does not simulate trading profits. The classification results do not include transaction costs, bid–ask spreads, slippage, taxes, or portfolio constraints, and they should not be presented as evidence of a profitable strategy.

# Part 2 — Search Volume Index as alternative data

*The assignment requires Part 2 only for groups of three.*

## 1. What it measures

Search Volume Index (SVI) data summarize the relative frequency of searches for a term or topic. Analysts use search activity as a proxy for public attention to a company, asset, or economic issue. Da, Engelberg, and Gao (2011) study search frequency as a measure of investor attention, while Preis, Moat, and Stanley (2013) examine Google Trends in relation to trading behavior.

Search activity can reflect investor attention, but it is not automatically a measure of positive or negative sentiment. A spike may reflect concern, a news event, curiosity, or an unrelated meaning of the search term. Any relationship with returns needs to be tested rather than assumed.

## 2. Sources and structure

Google Trends provides indices for search terms or topics, with options for geography, category, and time period. Baidu Index is another possible source for searches in Chinese-language contexts. Commercial providers may offer more detailed or higher-frequency data under a paid license.

SVI exports are usually relative indices rather than complete counts of searches. An index value of 100 generally identifies the peak in the selected comparison window; it does not mean 100 searches. When collecting data, record the term or topic, country or region, category, date range, frequency, provider, download date, and the file's license or terms of use.

For this notebook, a CSV file can be named `google_trends_svi.csv`. It should have a date column and a `search_index` column. An optional `asset_return` column can be used for exploratory comparisons.

## 3. Quality checks

Search indices can be revised or change across repeated downloads. Low-volume terms may contain missing values, and a zero may not mean that nobody searched. Terms can also be ambiguous, and daily search records must not be compared directly with weekly returns without aligning the dates.

Before analysis, check for duplicated dates, missing values, out-of-range values, consistent frequency, and the effects of query or geography choices. Preserve the raw download, document cleaning steps, and avoid treating a correlation as evidence of causation.

## 4. Ethical and licensing considerations

Use aggregate search data and avoid attempts to identify individuals or infer sensitive personal attributes. Follow the provider's terms, rate limits, licensing, attribution, and redistribution rules. Public accessibility does not necessarily permit unrestricted commercial reuse. Record the source and transformation steps so that the analysis can be reviewed.

## 5. Literature

Da, Engelberg, and Gao (2011) examine how search volume can capture investor attention. Preis, Moat, and Stanley (2013) explore the relationship between Google search activity and financial-market behavior. Together, these studies motivate testing search data as a complementary signal while leaving its predictive value to empirical evaluation.

The following cell reads a provided CSV. When no file is present, it creates a clearly labelled synthetic example so the import and EDA steps can still be demonstrated. Synthetic observations must not be described in the report as real Google Trends data.

In [ ]:
svi_file = Path("google_trends_svi.csv")

if svi_file.exists():
    svi = pd.read_csv(svi_file)
    column_lookup = {
        str(column).strip().lower().replace(" ", "_"): column
        for column in svi.columns
    }

    date_column = next(
        (column_lookup[name] for name in ("date", "week", "month") if name in column_lookup),
        None,
    )
    index_column = next(
        (
            column_lookup[name]
            for name in ("search_index", "search_volume_index", "svi")
            if name in column_lookup
        ),
        None,
    )
    return_column = next(
        (column_lookup[name] for name in ("asset_return", "ivv_return") if name in column_lookup),
        None,
    )

    if date_column is None or index_column is None:
        raise ValueError(
            "The CSV needs a date/week/month column and a search_index column."
        )

    rename_columns = {date_column: "date", index_column: "search_index"}
    if return_column is not None:
        rename_columns[return_column] = "asset_return"
    svi = svi.rename(columns=rename_columns)

    svi["date"] = pd.to_datetime(svi["date"], errors="coerce")
    svi["search_index"] = pd.to_numeric(svi["search_index"], errors="coerce")
    if "asset_return" in svi:
        svi["asset_return"] = pd.to_numeric(svi["asset_return"], errors="coerce")

    svi = (
        svi.dropna(subset=["date", "search_index"])
        .sort_values("date")
        .drop_duplicates("date")
    )
    svi_source = "CSV file supplied by the analyst"
else:
    # Synthetic data for demonstrating the workflow only.
    rng = np.random.default_rng(RANDOM_STATE)
    dates = pd.date_range("2023-01-01", periods=52, freq="W")
    search_index = np.clip(
        48 + 13 * np.sin(np.linspace(0, 4 * np.pi, len(dates))) + rng.normal(0, 7, len(dates)),
        0,
        100,
    )
    svi = pd.DataFrame({
        "date": dates,
        "search_index": search_index,
        "asset_return": rng.normal(0.001, 0.02, len(dates)),
    })
    svi_source = "Synthetic example — not observed search data"

print(f"Data source: {svi_source}")
display(svi.head())

columns_to_describe = ["search_index"]
if "asset_return" in svi.columns:
    columns_to_describe.append("asset_return")
display(svi[columns_to_describe].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(svi["date"], svi["search_index"], marker="o", markersize=3, linewidth=1.2)
axes[0].set_title("Search index over time")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Relative search index")
sns.histplot(svi["search_index"], bins=12, kde=True, ax=axes[1])
axes[1].set_title("Distribution of search index")
axes[1].set_xlabel("Search index")
axes[1].set_ylabel("Observations")
fig.tight_layout()
plt.show()

display(svi.isna().sum().rename("Missing values").to_frame())

if "asset_return" in svi.columns:
    aligned = svi[["search_index", "asset_return"]].dropna()
    if len(aligned) >= 3:
        print(
            "Contemporaneous correlation (exploratory only): "
            f"{aligned['search_index'].corr(aligned['asset_return']):.3f}"
        )
    else:
        print("There are too few aligned observations to calculate correlation.")

## Works Cited

Da, Zhi, Joseph Engelberg, and Pengjie Gao. “In Search of Attention.” *The Journal of Finance*, vol. 66, no. 5, 2011, pp. 1461–1499. https://doi.org/10.1111/j.1540-6261.2011.01679.x.

Google. “Google Trends.” *Google Trends*, https://trends.google.com/. Accessed 15 Oct. 2026.

Preis, Tobias, Helen Susannah Moat, and H. Eugene Stanley. “Quantifying Trading Behavior in Financial Markets Using Google Trends.” *Scientific Reports*, vol. 3, 2013, article 1684. https://doi.org/10.1038/srep01684.

Sagaceta-Mejía, Alma Rocío, Máximo Eduardo Sánchez-Gutiérrez, and Julián Alberto Fresán-Figueroa. “An Intelligent Approach for Predicting Stock Market Movements in Emerging Markets Using Optimized Technical Indicators and Neural Networks.” *Economics*, vol. 18, no. 1, 2024, article 20220073. https://doi.org/10.1515/econ-2022-0073.

Sun, et al. “Alternative Data in Finance and Business: Emerging Applications and Theory Analysis (Review).” *Financial Innovation*, vol. 10, 2024, article 127. https://doi.org/10.1186/s40854-024-00652-0.

Yahoo Finance. “iShares Core S&P 500 ETF (IVV) Historical Data.” *Yahoo Finance*, https://finance.yahoo.com/quote/IVV/history/. Accessed 15 Oct. 2026.

In the written report, cite sources in the paragraphs where their ideas or findings are discussed. The bibliography alone does not replace in-text citations.